In this Colab, you'll go from a raw pharmaceutical PDF blob to page-level classification, logical document grouping, chunking, embedding with metadata, and filtered retrieval.

By the end, your system will:

* Classify pharmaceutical documents like Certificates of Quality, Packaging Specifications, and BSE/TSE Declarations
* Retrieve only relevant chunks using smart filters like `doc_type`
* Show improved performance in noisy blob-style files containing multiple document types
* Be fully modular for use in production document intelligence pipelines

# Setup

In [1]:
# ============================================
# STEP 1: Install Required Packages
# ============================================
#
# WHAT WE'RE DOING:
# Installing LlamaIndex (RAG framework), HuggingFace embeddings,
# PyPDF2 (PDF text extraction), and Google Generative AI (Gemini LLM).
#
# WHY THIS MATTERS:
# This end-to-end pipeline combines all the tools from previous notebooks:
# PDF reading, LLM classification, embedding, and filtered retrieval.
#
# WHAT YOU'LL SEE:
# Package installation output (may take 1-2 minutes).
# ============================================

!pip install -q llama-index llama-index-readers-file llama-index-embeddings-huggingface
!pip install -q transformers sentence-transformers PyPDF2
!pip install -q pymupdf pandas langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25

In [2]:
# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 282.0 MB/s eta 0:00:00


In [21]:
!pip install -q llama-cpp-python

### Import libraries

In [3]:
import os
import json
import uuid
import fitz  # PyMuPDF
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

### Upload your contract PDF

In [4]:
from google.colab import files

uploaded = files.upload()

pdf_paths = list(uploaded.keys())

print("Uploaded files:")
for path in pdf_paths:
    print(path)

Saving pharma-blob-sample.pdf to pharma-blob-sample.pdf
Uploaded files:
pharma-blob-sample.pdf


### Set document metadata

In [5]:
doc_type = "Packaging Specification"
doc_id = "packaging_spec_01"

source_file = pdf_paths[0]

print("Using PDF:", source_file)

Using PDF: pharma-blob-sample.pdf


### Extract text from each page

In [6]:
page_metadata_store = []

doc = fitz.open(source_file)

for page_index, page in enumerate(doc):
    page_text = page.get_text("text")

    page_metadata_store.append({
        "page": page_index,
        "doc_type": doc_type,
        "text": page_text,
        "doc_id": doc_id,
        "source_file": source_file,
        "page_in_doc": page_index
    })

doc.close()

print("Total pages extracted:", len(page_metadata_store))
print(page_metadata_store[0])

Total pages extracted: 10
{'page': 0, 'doc_type': 'Packaging Specification', 'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription\nPart Number\nOperating Temperature\nHigh Flow Kit F, Modified, AKTA ready\n29477427\n+2°C to +40°C\n

### Reconstruct the full document

In [7]:
full_contract_text = "\n\n".join(
    page["text"] for page in page_metadata_store
)

print(full_contract_text[:1000])

Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits
as well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to
brittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to
+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, AKTA ready
29477427
+2°C to +40°C
High Flow Gradient C, Modified, AKTA ready
29184612
+2°C to +40°C
Operating Instructions access link:
https

### Apply sliding-window chunking

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)

chunks = splitter.split_text(full_contract_text)

print("Total chunks created:", len(chunks))
print(chunks[0])

Total chunks created: 28
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits
as well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to


### Attach metadata to each chunk

In [9]:
all_documents = []

for i, chunk in enumerate(chunks):
    all_documents.append(
        Document(
            text=chunk,
            metadata={
                "doc_type": doc_type,
                "chunk_index": i,
                "doc_id": doc_id,
                "source_file": source_file
            }
        )
    )

print("Total LlamaIndex documents:", len(all_documents))
print(all_documents[0].metadata)

Total LlamaIndex documents: 28
{'doc_type': 'Packaging Specification', 'chunk_index': 0, 'doc_id': 'packaging_spec_01', 'source_file': 'pharma-blob-sample.pdf'}


### Load open-source Mistral model

In [22]:
from llama_cpp import Llama
import os

model_path = "/content/mistral.gguf"  # change this to your model path
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model file exists. Size: 4166.07 MB


llm_load_tensors: offloading 1 repeating layers to GPU
llm_load_tensors: offloaded 1/33 layers to GPU
llm_load_tensors:        CPU buffer size =  4165.37 MiB
llm_load_tensors:      CUDA0 buffer size =   132.50 MiB
.................................................................................................
llama_new_context_with_model: n_ctx      = 2048
llama_new_context_with_model: n_batch    = 512
llama_new_context_with_model: n_ubatch   = 512
llama_new_context_with_model: flash_attn = 0
llama_new_context_with_model: freq_base  = 1000000.0
llama_new_context_with_model: freq_scale = 1
llama_kv_cache_init:  CUDA_Host KV buffer size =   248.00 MiB
llama_kv_cache_init:      CUDA0 KV buffer size =     8.00 MiB
llama_new_context_with_model: KV self size  =  256.00 MiB, K (f16):  128.00 MiB, V (f16):  128.00 MiB
llama_new_context_with_model:  CUDA_Host  output buffer size =     0.12 MiB
llama_new_context_with_model:      CUDA0 compute buffer size =   181.04 MiB
llama_new_context_with_mo

Model loaded successfully!


In [23]:
def local_llama_model(prompt, max_tokens=300, temperature=0.0):
    formatted_prompt = f"""
[INST]
{prompt}
[/INST]
"""

    response = llm(
        formatted_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["</s>", "[/INST]"]
    )

    return response["choices"][0]["text"].strip()

### Set up embedding model

In [12]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Create the vector index

In [13]:
index = VectorStoreIndex.from_documents(all_documents)

print("Vector index created successfully.")

Vector index created successfully.


### Define query and predicted document type

In [14]:
query = "Were there any packaging configuration changes?"

predicted_doc_type = "Packaging Specification"

print("Query:", query)
print("Predicted document type:", predicted_doc_type)

Query: Were there any packaging configuration changes?
Predicted document type: Packaging Specification


### Retrieve chunks using metadata filtering

In [15]:
retriever = index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="doc_type",
                value=predicted_doc_type,
                operator=FilterOperator.EQ
            )
        ]
    )
)

retrieved_nodes = retriever.retrieve(query)

print("Retrieved chunks:", len(retrieved_nodes))

for node in retrieved_nodes:
    print("=" * 80)
    print("Score:", node.score)
    print("Metadata:", node.metadata)
    print("Text:", node.text[:500])

Retrieved chunks: 4
Score: 0.7265216446618726
Metadata: {'doc_type': 'Packaging Specification', 'chunk_index': 12, 'doc_id': 'packaging_spec_01', 'source_file': 'pharma-blob-sample.pdf'}
Text: 5. Drawing Changes
Drawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2.
Key changes located on drawing:
- Sheet 1: Updated tray cavity dimensions (Section B-B)
- Sheet 2: New lid film sealing parameters (Detail C)
- Sheet 3: Updated label placement (per ECN-2023-0847)
Approved by: Packaging Engineering, Cytiva Uppsala
Date: 2023-11-08
Score: 0.7149186983141596
Metadata: {'doc_type': 'Packaging Specification', 'chunk_index': 26, 'doc_id': 'packaging_spec_01', 'source_file': 'pharma-blob-sample.pdf'}
Text: 29477427
High Flow Kit F, Modified, AKTA ready
29184612
High Flow Gradient C, Modified, AKTA ready
All listed assemblies follow the same chain of custody:
1. Raw materials received at Eysins facility (incoming QC)
2. Sub-assembly and final assembly (in-process checks)
3. Final quality rele

### Generate final answer from retrieved chunks

In [24]:
context = "\n\n".join(
    node.text for node in retrieved_nodes
)

prompt = f"""
You are answering a question using only the retrieved contract/PDF chunks below.

Question:
{query}

Retrieved context:
{context}

Instructions:
- Answer only from the retrieved context.
- If the answer is not present, say: "The answer was not found in the retrieved chunks."
- Be concise and specific.

Answer:
"""

answer = local_llama_model(prompt)

print("Answer:")
print(answer)


llama_print_timings:        load time =    1942.06 ms
llama_print_timings:      sample time =       3.66 ms /    80 runs   (    0.05 ms per token, 21840.02 tokens per second)
llama_print_timings: prompt eval time =    3169.75 ms /   783 tokens (    4.05 ms per token,   247.02 tokens per second)
llama_print_timings:        eval time =   51565.00 ms /    79 runs   (  652.72 ms per token,     1.53 tokens per second)
llama_print_timings:       total time =   54798.87 ms /   862 tokens


Answer:
Yes, there were packaging configuration changes. The blister material was changed from PVC to PETG (ECN-2023-0847). The drawing DWG-29477427-PKG-R3 supersedes the previous version, with updates including tray cavity dimensions, lid film sealing parameters, and label placement.


### Package the final output

In [25]:
final_output = {
    "query": query,
    "predicted_doc_type": predicted_doc_type,
    "matched_chunks": [
        {
            "text": node.text,
            "metadata": node.metadata,
            "score": float(node.score) if node.score is not None else None
        }
        for node in retrieved_nodes
    ],
    "answer": answer
}

import json

print(json.dumps(final_output, indent=4))

{
    "query": "Were there any packaging configuration changes?",
    "predicted_doc_type": "Packaging Specification",
    "matched_chunks": [
        {
            "text": "5. Drawing Changes\nDrawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2.\nKey changes located on drawing:\n- Sheet 1: Updated tray cavity dimensions (Section B-B)\n- Sheet 2: New lid film sealing parameters (Detail C)\n- Sheet 3: Updated label placement (per ECN-2023-0847)\nApproved by: Packaging Engineering, Cytiva Uppsala\nDate: 2023-11-08",
            "metadata": {
                "doc_type": "Packaging Specification",
                "chunk_index": 12,
                "doc_id": "packaging_spec_01",
                "source_file": "pharma-blob-sample.pdf"
            },
            "score": 0.7265216446618726
        },
        {
            "text": "29477427\nHigh Flow Kit F, Modified, AKTA ready\n29184612\nHigh Flow Gradient C, Modified, AKTA ready\nAll listed assemblies follow the same chain of custody

### Save result to JSON

In [26]:
with open("metadata_filtered_rag_output.json", "w", encoding="utf-8") as file:
    json.dump(final_output, file, indent=4, ensure_ascii=False)

print("Saved metadata_filtered_rag_output.json")

Saved metadata_filtered_rag_output.json


### Compare with single-page retrieval

In [27]:
single_page_documents = []

for page in page_metadata_store:
    single_page_documents.append(
        Document(
            text=page["text"],
            metadata={
                "doc_type": page["doc_type"],
                "doc_id": page["doc_id"],
                "source_file": page["source_file"],
                "page": page["page"],
                "page_in_doc": page["page_in_doc"]
            }
        )
    )

single_page_index = VectorStoreIndex.from_documents(single_page_documents)

single_page_retriever = single_page_index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(
        filters=[
            MetadataFilter(
                key="doc_type",
                value=predicted_doc_type,
                operator=FilterOperator.EQ
            )
        ]
    )
)

single_page_nodes = single_page_retriever.retrieve(query)

print("Single-page retrieval results:")

for node in single_page_nodes:
    print("=" * 80)
    print("Score:", node.score)
    print("Metadata:", node.metadata)
    print("Text:", node.text[:500])

Single-page retrieval results:
Score: 0.7075835789073529
Metadata: {'doc_type': 'Packaging Specification', 'doc_id': 'packaging_spec_01', 'source_file': 'pharma-blob-sample.pdf', 'page': 4, 'page_in_doc': 4}
Text: Packaging Component Specification (continued)
Document Number: PKG-SPEC-2023-0847, Page 2 of 2
4. Configuration Change History
Rev
Date
ECN Number
Change Description
A
2019-03-15
ECN-2019-0312
Initial release - PVC blister tray
B
2021-06-22
ECN-2021-0541
Updated tray insert dimensions per customer feedback (+3mm)
C
2023-11-08
ECN-2023-0847
Changed blister material from PVC to PETG (EU Reg 2023/1297)
5. Drawing Changes
Drawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2.
Key changes locate
Score: 0.6771625242691476
Metadata: {'doc_type': 'Packaging Specification', 'doc_id': 'packaging_spec_01', 'source_file': 'pharma-blob-sample.pdf', 'page': 3, 'page_in_doc': 3}
Text: Packaging Component Specification
Document Number:
PKG-SPEC-2023-0847
Revision:
C
Effective Date:
2023